In [1]:
# Cellule 1: Imports
import numpy as np
np.set_printoptions(threshold=10000, suppress=True)
import pandas as pd
import warnings
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, roc_curve, auc
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings('ignore')


In [2]:
# ============================================================================
# Cellule 2: Chargement et exploration des données
# ============================================================================

# Charger les données (sans header)
df = pd.read_csv('credit.data', header=None)

print("Shape du dataset:", df.shape)
print("\nPremières lignes:")
print(df.head(10))
print("\nTypes de données:")
print(df.dtypes)
print("\nNombre de valeurs manquantes ('?') par colonne:")
for i in range(df.shape[1]):
    count_missing = (df[i] == '?').sum()
    if count_missing > 0:
        print(f"Colonne {i}: {count_missing} valeurs manquantes")

Shape du dataset: (688, 1)

Premières lignes:
                                                   0
0  b\t30.83\t0\tu\tg\tw\tv\t1.25\tt\tt\t1\tf\tg\t...
1  a\t58.67\t4.46\tu\tg\tq\th\t3.04\tt\tt\t6\tf\t...
2  a\t24.50\t0.5\tu\tg\tq\th\t1.5\tt\tf\t0\tf\tg\...
3  b\t27.83\t1.54\tu\tg\tw\tv\t3.75\tt\tt\t5\tt\t...
4  b\t20.17\t5.625\tu\tg\tw\tv\t1.71\tt\tf\t0\tf\...
5  b\t32.08\t4\tu\tg\tm\tv\t2.5\tt\tf\t0\tt\tg\t3...
6  b\t33.17\t1.04\tu\tg\tr\th\t6.5\tt\tf\t0\tt\tg...
7  a\t22.92\t11.585\tu\tg\tcc\tv\t0.04\tt\tf\t0\t...
8  b\t54.42\t0.5\ty\tp\tk\th\t3.96\tt\tf\t0\tf\tg...
9  b\t42.50\t4.915\ty\tp\tw\tv\t3.165\tt\tf\t0\tt...

Types de données:
0    object
dtype: object

Nombre de valeurs manquantes ('?') par colonne:


In [3]:
# ============================================================================
# Cellule 3: Séparation des features et target
# ============================================================================

# Convertir en numpy array
data = df.values

# Séparer X et Y
X = data[:, :-1]  # Toutes les colonnes sauf la dernière
Y = data[:, -1]   # Dernière colonne (target)

print("Shape de X:", X.shape)
print("Shape de Y:", Y.shape)
print("\nClasses dans Y:", np.unique(Y))
print("Répartition des classes:")
unique, counts = np.unique(Y, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Classe '{u}': {c} instances ({c/len(Y)*100:.1f}%)")


Shape de X: (688, 0)
Shape de Y: (688,)

Classes dans Y: ['?\t20.08\t0.125\tu\tg\tq\tv\t1\tf\tt\t1\tf\tg\t240\t768\t+'
 '?\t20.42\t7.5\tu\tg\tk\tv\t1.5\tt\tt\t1\tf\tg\t160\t234\t+'
 '?\t24.50\t12.75\tu\tg\tc\tbb\t4.75\tt\tt\t2\tf\tg\t73\t444\t+'
 '?\t26.50\t2.71\ty\tp\t?\t?\t0.085\tf\tf\t0\tf\ts\t80\t0\t-'
 '?\t28.17\t0.585\tu\tg\taa\tv\t0.04\tf\tf\t0\tf\tg\t260\t1004\t-'
 '?\t29.50\t2\ty\tp\te\th\t2\tf\tf\t0\tf\tg\t256\t17\t-'
 '?\t29.75\t0.665\tu\tg\tw\tv\t0.25\tf\tf\t0\tt\tg\t300\t0\t-'
 '?\t32.25\t1.5\tu\tg\tc\tv\t0.25\tf\tf\t0\tt\tg\t372\t122\t-'
 '?\t33.17\t2.25\ty\tp\tcc\tv\t3.5\tf\tf\t0\tt\tg\t200\t141\t-'
 '?\t40.83\t3.5\tu\tg\ti\tbb\t0.5\tf\tf\t0\tf\ts\t1160\t0\t-'
 '?\t42.25\t1.75\ty\tp\t?\t?\t0\tf\tf\t0\tt\tg\t150\t1\t-'
 '?\t45.33\t1\tu\tg\tq\tv\t0.125\tf\tf\t0\tt\tg\t263\t0\t-'
 'a\t15.75\t0.375\tu\tg\tc\tv\t1\tf\tf\t0\tf\tg\t120\t18\t-'
 'a\t15.83\t0.585\tu\tg\tc\th\t1.5\tt\tt\t2\tf\tg\t100\t0\t+'
 'a\t15.83\t7.625\tu\tg\tq\tv\t0.125\tf\tt\t1\tt\tg\t0\t160\t-'
 'a\t15.92

In [4]:
# ============================================================================
# Cellule 4: Binarisation de la target
# ============================================================================

# Convertir + en 1 et - en 0
Y_binary = np.where(Y == '+', 1, 0)

print("Target binarisée:")
print("Classe 0 (-):", np.sum(Y_binary == 0))
print("Classe 1 (+):", np.sum(Y_binary == 1))

Target binarisée:
Classe 0 (-): 688
Classe 1 (+): 0


In [5]:
# ============================================================================
# Cellule 5: Identification des colonnes continues et catégorielles
# ============================================================================

# Selon la documentation UCI:
# Colonnes numériques (indices): 1, 2, 7, 10, 13, 14
# Colonnes catégorielles (indices): 0, 3, 4, 5, 6, 8, 9, 11, 12

col_num = [1, 2, 7, 10, 13, 14]
col_cat = [0, 3, 4, 5, 6, 8, 9, 11, 12]

print("Colonnes numériques:", col_num)
print("Colonnes catégorielles:", col_cat)

Colonnes numériques: [1, 2, 7, 10, 13, 14]
Colonnes catégorielles: [0, 3, 4, 5, 6, 8, 9, 11, 12]


In [6]:
# ============================================================================
# Cellule 6: Traitement des variables catégorielles
# ============================================================================

# Copier les colonnes catégorielles
X_cat = np.copy(X[:, col_cat])

print("Shape des données catégorielles:", X_cat.shape)
print("\nExemple de données catégorielles avant transformation:")
print(X_cat[:5])

# Transformer les valeurs catégorielles en valeurs numériques
for col_id in range(len(col_cat)):
    unique_val, val_idx = np.unique(X_cat[:, col_id], return_inverse=True)
    X_cat[:, col_id] = val_idx
    print(f"\nColonne {col_cat[col_id]}: {len(unique_val)} modalités uniques")

print("\nDonnées catégorielles après transformation:")
print(X_cat[:5])

IndexError: index 0 is out of bounds for axis 1 with size 0

In [ ]:
# ============================================================================
# Cellule 7: Imputation des valeurs manquantes - Variables catégorielles
# ============================================================================

# Remplacer '0' qui correspond à '?' par la valeur la plus fréquente
# Note: après unique(), '?' devient souvent la valeur 0
X_cat = X_cat.astype(float)

# Imputer avec la stratégie most_frequent
imp_cat = SimpleImputer(missing_values=0, strategy='most_frequent')
X_cat = imp_cat.fit_transform(X_cat)

print("Valeurs manquantes dans X_cat après imputation:", np.sum(np.isnan(X_cat)))
print("\nDonnées catégorielles après imputation:")
print(X_cat[:5])

In [ ]:
# Cellule 8: Traitement des variables numériques
# ============================================================================

# Copier les colonnes numériques
X_num = np.copy(X[:, col_num])

print("Shape des données numériques:", X_num.shape)
print("\nExemple de données numériques avant transformation:")
print(X_num[:5])

# Remplacer '?' par nan
X_num[X_num == '?'] = np.nan
X_num = X_num.astype(float)

print("\nNombre de valeurs manquantes par colonne numérique:")
for i in range(X_num.shape[1]):
    count_nan = np.sum(np.isnan(X_num[:, i]))
    print(f"Colonne {col_num[i]}: {count_nan} NaN")

In [ ]:
# ============================================================================
# Cellule 9: Imputation des valeurs manquantes - Variables numériques
# ============================================================================

# Imputer avec la moyenne
imp_num = SimpleImputer(missing_values=np.nan, strategy='mean')
X_num = imp_num.fit_transform(X_num)

print("Valeurs manquantes dans X_num après imputation:", np.sum(np.isnan(X_num)))
print("\nDonnées numériques après imputation:")
print(X_num[:5])
print("\nStatistiques des variables numériques:")
print("Moyennes:", np.mean(X_num, axis=0))
print("Écarts-types:", np.std(X_num, axis=0))

In [ ]:
# ============================================================================
# Cellule 10: Encodage One-Hot des variables catégorielles
# ============================================================================

# Encoder les variables catégorielles en variables binaires
encoder = OneHotEncoder(sparse_output=False)
X_cat_bin = encoder.fit_transform(X_cat)

print("Shape après One-Hot Encoding:", X_cat_bin.shape)
print("Nombre de features catégorielles binaires créées:", X_cat_bin.shape[1])
print("\nExemple de données après One-Hot Encoding:")
print(X_cat_bin[:5, :10])  # Afficher les 10 premières colonnes

In [ ]:
# ============================================================================
# Cellule 11: Normalisation des variables numériques
# ============================================================================

# Normaliser les variables continues
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

print("Données numériques après normalisation:")
print("Moyennes:", np.mean(X_num_scaled, axis=0))
print("Écarts-types:", np.std(X_num_scaled, axis=0))
print("\nExemple de données normalisées:")
print(X_num_scaled[:5])

In [ ]:
# ============================================================================
# Cellule 12: Construction du jeu de données final
# ============================================================================

# Concaténer les données catégorielles encodées et les données numériques normalisées
X_final = np.concatenate([X_cat_bin, X_num_scaled], axis=1)

print("Shape du jeu de données final:", X_final.shape)
print("Nombre total de features:", X_final.shape[1])
print("  - Features catégorielles (binaires):", X_cat_bin.shape[1])
print("  - Features numériques (normalisées):", X_num_scaled.shape[1])


In [ ]:
# ============================================================================
# Cellule 13: Séparation train/test
# ============================================================================

X_train, X_test, Y_train, Y_test = train_test_split(
    X_final, Y_binary, test_size=0.5, random_state=1
)

print("\nSéparation train/test:")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Y_train - Classe 0:", np.sum(Y_train == 0), "| Classe 1:", np.sum(Y_train == 1))
print("Y_test - Classe 0:", np.sum(Y_test == 0), "| Classe 1:", np.sum(Y_test == 1))

In [ ]:
# ============================================================================
# Cellule 14: Fonction d'évaluation
# ============================================================================

def evaluate_model(y_true, y_pred, y_proba, model_name):
    """
    Évalue un modèle et affiche les métriques
    """
    # Matrice de confusion
    cm = confusion_matrix(y_true, y_pred)
    
    # Métriques
    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    
    # Pour le crédit scoring, le rappel est plus important (détecter les défauts)
    best_metric = recall
    final_score = (acc + best_metric) / 2
    
    # Courbe ROC et AUC
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    roc_auc = auc(fpr, tpr)
    
    # Affichage
    print(f"\n{'='*60}")
    print(f"Résultats pour: {model_name}")
    print(f"{'='*60}")
    print(f"Matrice de confusion:")
    print(cm)
    print(f"\nAccuracy: {acc:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"AUC: {roc_auc:.4f}")
    print(f"Score final (Accuracy + Recall)/2: {final_score:.4f}")
    
    # Graphique ROC
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {model_name}')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()
    
    return final_score, roc_auc

In [ ]:
# ============================================================================
# Cellule 15: Fonction run_classifiers
# ============================================================================

def run_classifiers(X_train, X_test, Y_train, Y_test, classifiers_dict):
    """
    Entraîne et évalue plusieurs classifieurs
    """
    results = {}
    
    for name, clf in classifiers_dict.items():
        print(f"\n\nEntraînement de {name}...")
        
        # Entraînement
        clf.fit(X_train, Y_train)
        
        # Prédictions
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        
        # Évaluation
        score, roc_auc = evaluate_model(Y_test, y_pred, y_proba, name)
        
        results[name] = {
            'model': clf,
            'score': score,
            'auc': roc_auc
        }
    
    # Meilleur modèle
    best_model_name = max(results, key=lambda x: results[x]['score'])
    print(f"\n\n{'='*60}")
    print(f"MEILLEUR MODÈLE: {best_model_name}")
    print(f"Score final: {results[best_model_name]['score']:.4f}")
    print(f"AUC: {results[best_model_name]['auc']:.4f}")
    print(f"{'='*60}")
    
    return results, results[best_model_name]['model']

In [ ]:
# ============================================================================
# Cellule 16: Exécution sur les données préparées
# ============================================================================

# Définir les classifieurs
classifiers = {
    'CART': DecisionTreeClassifier(random_state=1),
    'KNN-5': KNeighborsClassifier(n_neighbors=5),
    'MLP': MLPClassifier(hidden_layer_sizes=(40, 20), random_state=1, max_iter=500)
}

print("ÉVALUATION SUR DONNÉES COMPLÈTES (catégorielles + numériques)")
print("="*60)

results, best_model = run_classifiers(X_train, X_test, Y_train, Y_test, classifiers)

In [ ]:
# ============================================================================
# Cellule 17: Résumé des résultats
# ============================================================================

print("\n\nRÉSUMÉ DES RÉSULTATS")
print("="*60)
print(f"{'Modèle':<15} {'Score Final':<15} {'AUC':<10}")
print("-"*60)
for name, res in results.items():
    print(f"{name:<15} {res['score']:<15.4f} {res['auc']:<10.4f}")

print("\n\nCONCLUSION:")
print("-"*60)
print(f"• Nombre total d'instances: {len(X_final)}")
print(f"• Nombre de features finales: {X_final.shape[1]}")
print(f"  - Variables catégorielles encodées: {X_cat_bin.shape[1]}")
print(f"  - Variables numériques normalisées: {X_num_scaled.shape[1]}")
print(f"• Données manquantes: traitées par imputation")
print(f"• Meilleur modèle: {max(results, key=lambda x: results[x]['score'])}")